In [1]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.inputs_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids)- max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.inputs_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.inputs_ids)

    def __getitem__(self, idx):
        return self.inputs_ids[idx], self.target_ids[idx]

In [2]:
import tiktoken
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset,
                            batch_size=batch_size,
                            shuffle=shuffle,
                            drop_last=drop_last,   # Drop the last batch if it's smaller than the specified batch size
                            num_workers=num_workers) # Number of subprocesses to use for data loading. 0 means that the data will be loaded in the main process.
    return dataloader

In [4]:
with open("../data/verdict.txt" ,"r", encoding="utf-8")   as f:
    raw_txt = f.read()

dataloader = create_dataloader_v1(raw_txt, batch_size=3, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader) # Convert the dataloader to an iterator to access the batches
first_batch = next(data_iter) # Get the first batch from the iterator
print("Input IDs:", first_batch)

Input IDs: [tensor([[  40,  367, 2885, 1464],
        [ 367, 2885, 1464, 1807],
        [2885, 1464, 1807, 3619]]), tensor([[ 367, 2885, 1464, 1807],
        [2885, 1464, 1807, 3619],
        [1464, 1807, 3619,  402]])]


In [7]:
secoond_batch = next(data_iter) # Get the second batch from the iterator
print("Input IDs:", secoond_batch)

Input IDs: [tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [8]:
dataloader2 = create_dataloader_v1(raw_txt, batch_size=1, max_length=2, stride=2, shuffle=False)
data_iter2 = iter(dataloader2) # Convert the dataloader to an iterator to access the batches
first_batch2 = next(data_iter2) # Get the first batch from the iterator
print("Input IDs:", first_batch2)
second_batch2 = next(data_iter2) # Get the second batch from the iterator
print("Input IDs:", second_batch2)

Input IDs: [tensor([[ 40, 367]]), tensor([[ 367, 2885]])]
Input IDs: [tensor([[2885, 1464]]), tensor([[1464, 1807]])]


In [9]:
dataloader3 = create_dataloader_v1(raw_txt, batch_size=4, max_length=4, stride=2, shuffle=False)
data_iter3 = iter(dataloader3) # Convert the dataloader to an iterator to access the batches
first_batch3 = next(data_iter3) # Get the first batch from the iterator
print("Input IDs:", first_batch3)
second_batch3 = next(data_iter3) # Get the second batch from the iterator
print("Input IDs:", second_batch3)

Input IDs: [tensor([[   40,   367,  2885,  1464],
        [ 2885,  1464,  1807,  3619],
        [ 1807,  3619,   402,   271],
        [  402,   271, 10899,  2138]]), tensor([[  367,  2885,  1464,  1807],
        [ 1464,  1807,  3619,   402],
        [ 3619,   402,   271, 10899],
        [  271, 10899,  2138,   257]])]
Input IDs: [tensor([[10899,  2138,   257,  7026],
        [  257,  7026, 15632,   438],
        [15632,   438,  2016,   257],
        [ 2016,   257,   922,  5891]]), tensor([[ 2138,   257,  7026, 15632],
        [ 7026, 15632,   438,  2016],
        [  438,  2016,   257,   922],
        [  257,   922,  5891,  1576]])]


#### Stride is the step size for moving the window across the tokenized text. It determines how much overlap there is between consecutive chunks of text. A smaller stride will result in more overlap, while a larger stride will result in less overlap. For example, if max_length is 4 and stride is 2, the first chunk will be tokens [0, 1, 2, 3], the second chunk will be tokens [2, 3, 4, 5], and so on.

In [18]:
input_ids = torch.tensor([2,3,5,1])
print("Input IDs shape :", input_ids.shape)

vocab_size = 6
output_dim = 3

torch.manual_seed(123) # Set a random seed for reproducibility
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print("Embedding layer weights:\n", embedding_layer.weight)

print(embedding_layer(torch.tensor([3])))
print(embedding_layer(input_ids))

Input IDs shape : torch.Size([4])
Embedding layer weights:
 Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [20]:
vocab_size = 50257 # GPT-2's vocabulary size
output_dim = 768 # Common embedding dimension for transformer models
token_embeddings_layer = torch.nn.Embedding(vocab_size, output_dim)

In [19]:
max_length = 4
dataloader = create_dataloader_v1(raw_txt, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print("Input IDs shape:", first_batch[0].shape) # Shape of input IDs
print("Target IDs shape:", first_batch[1].shape) # Shape of target IDs
print("Input IDs:\n", first_batch[0]) # Print the input IDs

Input IDs shape: torch.Size([8, 4])
Target IDs shape: torch.Size([8, 4])
Input IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])


In [22]:
token_embeddings = token_embeddings_layer(first_batch[0])
print("Token embeddings shape:", token_embeddings.shape) # Shape of the token embeddings

Token embeddings shape: torch.Size([8, 4, 768])


In [23]:
context_length = max_length
pos_embeddings_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embeddings_layer(torch.arange(context_length))
print("Positional embeddings shape:", pos_embeddings.shape) # Shape of the positional embeddings

Positional embeddings shape: torch.Size([4, 768])


In [25]:
input_embeddings = token_embeddings + pos_embeddings.unsqueeze(0) # Add positional embeddings to token embeddings
print("Input embeddings shape:", input_embeddings.shape) # Shape of the combined input embeddings

Input embeddings shape: torch.Size([8, 4, 768])


In [26]:
pos_embeddings.unsqueeze(0) 

tensor([[[ 0.9135,  1.3300, -0.4653,  ...,  1.6938,  1.3216, -1.4152],
         [-0.0655,  0.3985,  0.3842,  ..., -1.2460, -0.4325, -0.9073],
         [-1.2796,  0.6160, -1.0969,  ..., -1.2585,  0.6564, -1.4168],
         [ 0.0072,  0.7133,  0.8030,  ..., -1.0883,  0.7227, -0.1036]]],
       grad_fn=<UnsqueezeBackward0>)